# SOXX 官方历史份额：申赎状态来源预检

只审核 NAS 备份的发行人历史工作簿与原生 Qlib 日历，不读取市场结果、拟合模型或运行账户。冻结边界见 `plan.json`。

In [1]:
import os,sys,json,hashlib,re,datetime,math,tarfile,tempfile,subprocess,shlex,collections
from pathlib import Path
from xml.etree import ElementTree as ET
import pandas as pd,numpy as np,qlib
from qlib.data import D
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings
card=root/'research/stage1_soxx_official_share_creation_source_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen']
sha=lambda b:hashlib.sha256(b).hexdigest()
assert not (card/'run_state.json').exists(),'frozen source read already attempted'
assert sha((root/src['Qlib_provider']/'qlib.yaml').read_bytes())==src['Qlib_config_sha256']
assert sha((root/src['source_audit']).read_bytes())==src['source_audit_sha256']
receipt=json.loads((root/src['NAS_receipt']).read_text());assert receipt['verified'] and receipt['sha256']==src['NAS_archive_sha256']
assert receipt['members'][src['original_workbook']]==src['workbook_sha256']
state={'plan_sha256':sha((card/'plan.json').read_bytes()),'phase':'source_read_attempted','NAS_scoped_restore_attempts':1,'original_workbook_raw_read_attempts':1,'Qlib_D_calendar_call_attempts':1,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-share-source-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with archive.open('wb') as out:
  run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(receipt['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive.read_bytes())==src['NAS_archive_sha256']
 with tarfile.open(archive,'r:gz') as z:
  assert src['original_workbook'] in z.getnames()
  assert sha(z.extractfile(src['original_workbook']).read())==src['workbook_sha256']
raw=(root/src['original_workbook']).read_bytes();assert sha(raw)==src['workbook_sha256']
original=raw.decode('utf-8');repaired,amp_count=re.subn(r'&(?!amp;|lt;|gt;|quot;|apos;|#[0-9]+;|#x[0-9A-Fa-f]+;)', '&amp;',original)
assert amp_count==6
xml=ET.fromstring(repaired);ns={'s':'urn:schemas-microsoft-com:office:spreadsheet'};key='{urn:schemas-microsoft-com:office:spreadsheet}Name'
sheets={w.attrib[key]:w.findall('.//s:Row',ns) for w in xml.findall('s:Worksheet',ns)}
assert list(sheets)==['Disclaimers','Holdings','Historical','Performance','Distributions']
rowvals=lambda row:[c.findtext('s:Data',default='',namespaces=ns) for c in row.findall('s:Cell',ns)]
assert rowvals(sheets['Historical'][0])==['As Of','NAV per Share','Ex-Dividends','Shares Outstanding']
observations=[]
for row in sheets['Historical'][1:]:
 vals=rowvals(row);assert len(vals)==4
 d=pd.Timestamp(datetime.datetime.strptime(vals[0],'%b %d, %Y').date());s=vals[3].replace(',','').strip()
 try:v=float(s) if s else np.nan
 except ValueError:v=np.nan
 observations.append((d,v,vals[3]))
assert len(observations)==8937 and len({x[0] for x in observations})==8937
frame=pd.DataFrame(observations,columns=['date','shares','original_text']).set_index('date').sort_index()
assert str(frame.index.min().date())=='2001-07-10' and str(frame.index.max().date())=='2026-10-21'
qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
cal=pd.DatetimeIndex(D.calendar(start_time=src['decision_calendar_start'],end_time=src['decision_calendar_end'],freq='day'))
train=cal[cal<pd.Timestamp('2023-01-01')];held=cal[cal>=pd.Timestamp('2023-01-01')]
aligned=frame.reindex(cal);valid=np.isfinite(aligned.shares.to_numpy(float)) & (aligned.shares.to_numpy(float)>0)
changes=aligned.shares.diff();ratio=aligned.shares/aligned.shares.shift(1)
years={}
for y in range(2011,2027):
 z=aligned.loc[str(y)] if str(y) in aligned.index.strftime('%Y') else aligned.iloc[:0]
 good=z.shares[np.isfinite(z.shares) & (z.shares>0)]
 years[str(y)]={'qlib_sessions':len(z),'valid_share_rows':len(good),'first_shares':float(good.iloc[0]) if len(good) else None,'last_shares':float(good.iloc[-1]) if len(good) else None,'changed_sessions':int(z.shares.diff().fillna(0).ne(0).sum())}
large=ratio[(ratio>2)|(ratio<0.5)].dropna();large_rows=[{'date':str(d.date()),'ratio':float(v),'previous_shares':float(aligned.shares.shift(1).loc[d]),'shares':float(aligned.shares.loc[d])} for d,v in large.items()]
source_report={'decision':'historical_share_series_observed_PIT_and_split_semantics_unqualified','plan_sha256':sha((card/'plan.json').read_bytes()),'source_sha256':src['workbook_sha256'],'NAS_restored_sha256':src['NAS_archive_sha256'],'raw_workbook_reads':1,'Qlib_calendar_calls':1,'workbook_rows':len(frame),'workbook_first':str(frame.index.min().date()),'workbook_last':str(frame.index.max().date()),'nonfinite_workbook_rows':int((~np.isfinite(frame.shares)).sum()),'nonpositive_workbook_rows':int((frame.shares<=0).sum()),'fractional_positive_workbook_rows':int(((frame.shares>0)&(frame.shares%1!=0)).sum()),'Qlib_sessions':len(cal),'train_2011_2022_sessions':len(train),'held_2023_2026_sessions':len(held),'valid_Qlib_sessions':int(valid.sum()),'train_valid_sessions':int(valid[:len(train)].sum()),'held_valid_sessions':int(valid[len(train):].sum()),'years':years,'large_adjacent_QLib_share_ratios':large_rows,'weekend_workbook_rows':int((frame.index.weekday>=5).sum()),'first_publication_time_qualified':False,'historical_vintage_qualified':False,'split_adjustment_policy_qualified':False,'tradable_share_creation_feature_qualified':False,'SOXX_market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(source_report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':source_report['decision'],'objective_achieved':False,'plan_sha256':source_report['plan_sha256'],'result':'result.json','source_raw_NAS_sha256':src['NAS_archive_sha256'],'source_NAS_scoped_restore_verified':True,'model_fits':0,'native_accounts':0,'SOXX_market_outcome_reads':0,'artifact_policy':plan['artifact_policy'],'limitations':['Current workbook is a retrospective revision, not a historical first-publication archive.','Share count split-adjustment semantics must be checked before creation flow calculation.','Numeric coverage is source evidence only, not a Stage1 factor or SOXX performance result.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print(json.dumps({k:source_report[k] for k in ('decision','workbook_rows','Qlib_sessions','train_valid_sessions','held_valid_sessions','nonfinite_workbook_rows','nonpositive_workbook_rows','fractional_positive_workbook_rows','large_adjacent_QLib_share_ratios')},indent=2))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[34335:MainThread](2026-10-23 08:40:32,131) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[34335:MainThread](2026-10-23 08:40:32,461) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[34335:MainThread](2026-10-23 08:40:32,462) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{
  "decision": "historical_share_series_observed_PIT_and_split_semantics_unqualified",
  "workbook_rows": 8937,
  "Qlib_sessions": 3966,
  "train_valid_sessions": 3020,
  "held_valid_sessions": 946,
  "nonfinite_workbook_rows": 2,
  "nonpositive_workbook_rows": 0,
  "fractional_positive_workbook_rows": 0,
  "large_adjacent_QLib_share_ratios": []
}


# SOXX 官方历史流通份额：来源预检

以冻结完整 Stage1 0bp 账户为最终父本，本卡仅检查与价格、财报和披露权重不同的申购/赎回需求信息。复用 iShares 官方 `Historical` 工作簿原字节 SHA-256 `74265a67…`，其 Yanxuan NAS 归档 SHA-256 `a75d8e98…` 已再次限定恢复、逐字节校验。没有新 HTTP 请求、市场结果读取、模型拟合或账户。

工作簿共有 8,937 个唯一日期，其中两行非数值份额位于 Stage1 日历之外；原生 Qlib 日历的 2011–2022 训练 3,020 日和 2023–2026 复用历史 946 日，**全部有正整数份额值**。各年的交易日覆盖与变动次数见 `result.json`。覆盖充分只证明当前下载可解析，不证明值在当日已公布。2011–2026 全部相邻 Qlib 份额比均未超过 2 倍或低于 0.5 倍；[iShares 官方招募说明书](https://www.ishares.com/us/literature/prospectus/p-ishares-phlx-semiconductor-etf-3-31.pdf)记载 SOXX 于 2024-03-06 收盘后 3 比 1 拆分。两者**提示**当前工作簿历史份额经过回溯拆分标准化；尚未证明每个历史时点的原始份额、调整规则或首次公开时间。

**决定：`historical_share_series_observed_PIT_and_split_semantics_unqualified`。** 不据此计算申赎因子、运行信息筛选或 0bp 账户，更不能声称超过 SOXX。后续须取得发行人原始历史发布日期/版本或明确的每日发布政策，并用原始拆分事件核对跨期比例；通过来源门后，才可另行冻结“前一已发布交易日份额变化”与原 Stage1 39 项的匹配信息门，随后仍需完整自有状态的 0bp Tiered 费用账户及合同 v2 其余门槛。
